**SQL Cleaning Data **

In [0]:
%sql
select * from projects.portfolioprojects.nashvillehousing

-- standardize date format 

In [0]:
%sql
select SaleDate, CAST(SaleDate AS DATE)
from projects.portfolioprojects.nashvillehousing;





In [0]:
%sql
alter table projects.portfolioprojects.nashvillehousing
ADD COLUMN SaleDateConverted date;
    
update projects.portfolioprojects.nashvillehousing
set SaleDateConverted = CAST(SaleDate AS DATE);
    
select SaleDate, SaleDateConverted
from projects.portfolioprojects.nashvillehousing

-- populate property address data

In [0]:
%sql
select * 
from projects.portfolioprojects.nashvillehousing 
--where PropertyAddress is null
order by ParcelID

In [0]:
%sql
select a.ParcelID , a.PropertyAddress, b.ParcelID, b.PropertyAddress, coalesce(a.PropertyAddress, b.PropertyAddress)
from projects.portfolioprojects.nashvillehousing a
join projects.portfolioprojects.nashvillehousing b
on a.ParcelID = b.ParcelID and a.`UniqueID ` <> b.`UniqueID `
where a.PropertyAddress is null

ParcelID,PropertyAddress,ParcelID,PropertyAddress,"coalesce(a.PropertyAddress,b.PropertyAddress)"


In [0]:
%sql
merge into projects.portfolioprojects.nashvillehousing as tgt
using (
  select a.`UniqueID ` as uid, max(b.PropertyAddress) as new_address
  from projects.portfolioprojects.nashvillehousing a
      join projects.portfolioprojects.nashvillehousing b
      on a.ParcelID = b.ParcelID and a.`UniqueID ` <> b.`UniqueID `
  where a.PropertyAddress is null
  group by a.`UniqueID `
) as src
on tgt.`UniqueID ` = src.uid
when matched then update set tgt.PropertyAddress = src.new_address

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
29,29,0,0


breaking out address into individual columns (Address, city ,state )

In [0]:
%sql
select PropertyAddress
from projects.portfolioprojects.nashvillehousing 
--where PropertyAddress is null
--order by ParcelID

In [0]:
%sql
select 
substring(PropertyAddress, 1, charindex(',',PropertyAddress)-1) as Address,
substring(PropertyAddress, charindex(',',PropertyAddress)+1, len(PropertyAddress) ) as State
from projects.portfolioprojects.nashvillehousing

In [0]:
%sql
alter table projects.portfolioprojects.nashvillehousing
ADD COLUMN PropertySplitAddress varchar(255);
    
update projects.portfolioprojects.nashvillehousing
set PropertySplitAddress = substring(PropertyAddress, 1, charindex(',',PropertyAddress)-1);

alter table projects.portfolioprojects.nashvillehousing
ADD COLUMN PropertySplitCity varchar(255);
    
update projects.portfolioprojects.nashvillehousing
set PropertySplitCity = substring(PropertyAddress, charindex(',',PropertyAddress)+1, len(PropertyAddress) );

num_affected_rows
56477


In [0]:
%sql
select 
split(OwnerAddress, ',')[0] as address
, split(OwnerAddress, ',')[1] as city
, split(OwnerAddress, ',')[2] as state
from projects.portfolioprojects.nashvillehousing

In [0]:
%sql
alter table projects.portfolioprojects.nashvillehousing
ADD COLUMN OwnerSplitAddress varchar(255);
    
update projects.portfolioprojects.nashvillehousing
set OwnerSplitAddress = split(OwnerAddress, ',')[0];

alter table projects.portfolioprojects.nashvillehousing
ADD COLUMN OwnerSplitCity varchar(255);
    
update projects.portfolioprojects.nashvillehousing
set OwnerSplitCity = split(OwnerAddress, ',')[1];

alter table projects.portfolioprojects.nashvillehousing
ADD COLUMN OwnerSplitState varchar(255);
    
update projects.portfolioprojects.nashvillehousing
set OwnerSplitState = split(OwnerAddress, ',')[2]

num_affected_rows
56477


In [0]:
%sql
select *
from projects.portfolioprojects.nashvillehousing

change Y and N to yes and No in "Sold as vacant" column

In [0]:
%sql
select distinct(SoldAsVacant), count(SoldAsVacant)
from projects.portfolioprojects.nashvillehousing
group by SoldAsVacant
order by 2

SoldAsVacant,count(SoldAsVacant)
Yes,4675
No,51802


In [0]:
%sql

select SoldAsVacant, 
Case when SoldAsVacant = 'Y' then 'Yes'
     when SoldAsVacant = 'N' then 'No'
     else SoldAsVacant end
from projects.portfolioprojects.nashvillehousing

In [0]:
%sql
update projects.portfolioprojects.nashvillehousing
set SoldAsVacant = Case when SoldAsVacant = 'Y' then 'Yes'
     when SoldAsVacant = 'N' then 'No'
     else SoldAsVacant end

num_affected_rows
56477


remove duplicates 


In [0]:
%sql
with RowNumCTE as (
select * ,
    row_number() over (
        partition by ParcelID, 
        PropertyAddress,
        SalePrice, 
        SaleDate, 
        LegalReference
        order by `UniqueID `
    ) rownum
from projects.portfolioprojects.nashvillehousing
--order by ParcelID
)
select *
from rownumcte
where rownum > 1
order by PropertyAddress

UniqueID,ParcelID,LandUse,PropertyAddress,SaleDate,SalePrice,LegalReference,SoldAsVacant,OwnerName,OwnerAddress,Acreage,TaxDistrict,LandValue,BuildingValue,TotalValue,YearBuilt,Bedrooms,FullBath,HalfBath,SaleDateConverted,PropertySplitAddress,PropertySplitCity,OwnerSplitAddress,OwnerSplitCity,OwnerSplitState,rownum


In [0]:
%sql
merge into projects.portfolioprojects.nashvillehousing as tgt
using (
  select d.`UniqueID ` as uid
  from (
    select `UniqueID `,
        row_number() over (
            partition by ParcelID, 
            PropertyAddress,
            SalePrice, 
            SaleDate, 
            LegalReference
            order by `UniqueID `
        ) rownum
    from projects.portfolioprojects.nashvillehousing
    --order by ParcelID
  ) d
  where d.rownum > 1
) as src
on tgt.`UniqueID ` = src.uid
when matched then delete
--order by PropertyAddress

Delete unused columns 

In [0]:
%sql
Alter table projects.portfolioprojects.nashvillehousing
drop column OwnerAddress, PropertyAddress, TaxDistrict

In [0]:
%sql
select * from projects.portfolioprojects.nashvillehousing